# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Lane 2: Refresh / Content Opportunity Scoring

I'm chosing Lane 2 because I'm extending something I already understand, not starting something cold.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

This project helps the content editor decide which pages to review first  when they only have time to check a handful each week. The action taken is a content refresh (updating, expanding, or rewriting the page). A wrong recommendation costs either wasted editor time (if a fine page is flagged) or a missed real decline (if a genuinely worsening page isn't flagged and keeps losing traffic unnoticed) The unit of analysis is one page. A simple fixed rule (e.g. 'flag anything not updated in 6 months') would miss pages that are declining for other reasons, and would flag old pages that are still doing fine — data-driven scoring can weigh multiple signals together instead of relying on just one threshold

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess

In [2]:
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.head)



<bound method NDFrame.head of                  content_id          client_id  search_volume  competition  \
0      content_304f48230142  client_f369cb89fc           10.0         0.67   
1      content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2      content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3      content_331d6c4de07b  client_19581e27de           10.0         0.00   
4      content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   
...                     ...                ...            ...          ...   
29995  content_c322796023c8  client_e29c9c180c           10.0         0.05   
29996  content_526572edb3fa  client_7f2253d7e2            0.0         0.00   
29997  content_38112bdd0c6e  client_349c41201b           10.0         1.00   
29998  content_ab26273a7e7a  client_19581e27de           10.0         0.00   
29999  content_887020f20b5e  client_6208ef0f77            0.0         0.00   

      competition_level   cpc    

In [4]:
# How many pages are declining but stll have real search..
count1 = df[(df.trend_direction =="down") & (df.impressions_90d >= 100)]
print("Declining pages with real demand:", len(count1))

# How many pages are old but stil getiing traffic
count2 = df[(df.days_since_last_update >= 180) & (df.impressions_90d >= 500)]
print("Stale but visible pages:", len(count2 ))

#  what fraction of the whole dataset that represents
print("% of total pages:", round(len(count1) / len(df) * 100, 1), "%")

Declining pages with real demand: 13152
Stale but visible pages: 17
% of total pages: 43.8 %


Out of 30,000 pages, 13,152 (43.8%) are declining while still getting real search demand — nearly half the inventory is a candidate for review. Only 17 pages are both stale (unedited 180+ days) and still highly visible, showing that staleness alone is rare — declining-with-demand is the stronger, more common signal worth prioritizing.

In [5]:
print(df["days_since_last_update"].describe())

count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

My results will show which pages look like they need review, based on patterns in past data. This is decision-support only — it helps a human prioritize, it doesn't guarantee anything. I will not claim that refreshing a page causes traffic to recover, because proving cause-and-effect needs a real experiment, not just historical data. I also won't claim anything about how Google's algorithm works or how AI search tools rank pages, because my data can't show me that.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.